# 03 — Biopython Fundamentals

Biopython is a collection of Python tools designed for computational biology and bioinformatics.

This notebook introduces the core Biopython objects and operations that form the foundation for sequence analysis workflows:

- `Seq`
- `MutableSeq`
- `SeqRecord`
- biological sequence transformations
- DNA, RNA, and protein operations
- translation and codon tables
- metadata and annotations
- practical sequence-processing workflows
- reusable functions
- exercises and a mini-project
- a practical Biopython Cheat Sheet

> **Learning philosophy:** understand the biological concept, learn the Biopython abstraction, implement it, inspect the result, and understand its limitations.

## Learning Objectives

By the end of this notebook, you should be able to:

1. Explain what Biopython is and why it is useful.
2. Create and manipulate `Seq` objects.
3. Perform DNA/RNA/protein sequence operations.
4. Calculate complements and reverse complements.
5. Transcribe DNA into RNA and back-transcribe RNA into DNA.
6. Translate nucleotide sequences into proteins.
7. Understand translation options such as `to_stop`, `cds`, and codon tables.
8. Use `MutableSeq` when direct sequence editing is required.
9. Store biological sequence metadata using `SeqRecord`.
10. Build reusable Biopython-based sequence-analysis functions.
11. Recognize when a simple sequence operation is **not** a substitute for a real biological algorithm.
12. Use the final cheat sheet as a quick Biopython reference.

> **Code style:** Code cells use concise English comments to explain the purpose of important operations. The comments are intentionally focused on the biological/computational logic rather than explaining every Python token.


## 1. What Is Biopython?

Biopython is an open-source collection of Python modules for computational biology.

Instead of representing every biological sequence as a plain Python string, Biopython provides domain-specific objects and utilities.

For example:

```python
from Bio.Seq import Seq

dna = Seq("ATGCGTACG")
```

Now `dna` is a biological sequence object with operations such as:

- `reverse_complement()`
- `complement()`
- `transcribe()`
- `translate()`

Biopython also provides tools for:

- biological file formats
- sequence records and annotations
- alignments
- NCBI access
- BLAST
- protein analysis
- restriction enzymes
- phylogenetics
- biological databases

This notebook focuses on the **fundamental sequence objects**. File formats, NCBI, BLAST, alignment, and phylogenetics will be handled in later notebooks.

## 2. Installation and Version Check

Install Biopython with:

```bash
pip install biopython
```

For reproducible projects, always check the installed version.

In [1]:
import Bio

print("Biopython version:", Bio.__version__)

Biopython version: 1.86


If Biopython is not installed in your environment, run the installation command in a terminal or notebook cell first.

> **Version note:** Biopython APIs can evolve. When a project depends on a particular API, record the Biopython version used for the analysis.

### API compatibility note

This notebook targets the current Biopython API documented by the official Biopython documentation (Biopython 1.88 at the time this notebook was reviewed). Older tutorials may contain APIs that are no longer available in current releases. For example, older examples use `MutableSeq.toseq()`, while this notebook uses the current conversion pattern `Seq(mutable)`.

Official reference: https://biopython.org/docs/latest/


## 3. Core Biopython Architecture

The most important modules for this notebook are:

| Module | Main purpose |
|---|---|
| `Bio.Seq` | Biological sequence objects |
| `Bio.SeqRecord` | Sequences plus biological metadata |
| `Bio.SeqIO` | Reading and writing biological sequence files |
| `Bio.SeqFeature` | Genomic features and annotations |
| `Bio.SeqUtils` | Sequence and molecular utilities |
| `Bio.Data.CodonTable` | Genetic code and codon tables |
| `Bio.Align` | Sequence alignment |
| `Bio.Entrez` | Access to NCBI Entrez services |
| `Bio.Blast` | BLAST result handling |
| `Bio.Phylo` | Phylogenetic trees |

A useful mental model is:

```text
Seq
 ↓
SeqRecord
 ↓
SeqIO
 ↓
Biological files / datasets
 ↓
Alignment / Annotation / Databases / Analysis
```

# 4. The `Seq` Object

The `Seq` class is one of the most important objects in Biopython.

It represents a biological sequence while providing sequence-oriented methods.

In [2]:
from Bio.Seq import Seq

# Create a Biopython Seq object from a DNA sequence.
dna = Seq("ATGCGTACGTTAGC")

# Display the sequence, its Python type, and its length.
print(dna)
print(type(dna))
print(len(dna))

ATGCGTACGTTAGC
<class 'Bio.Seq.Seq'>
14


### Why not just use a Python string?

A Python string is excellent for general text manipulation.

A `Seq` object adds biological operations directly to the sequence:

```python
dna.reverse_complement()
dna.transcribe()
dna.translate()
```

This makes code easier to read as biological code rather than generic string manipulation.

## 5. Indexing and Slicing `Seq`

`Seq` objects support familiar Python indexing and slicing.

In [3]:
from Bio.Seq import Seq

# Create a DNA sequence for indexing and slicing examples.
dna = Seq("ATGCGTACGTTAGC")

# Access individual bases and subsequences using standard Python syntax.
print("First base:", dna[0])
print("Last base:", dna[-1])
print("First three:", dna[:3])
print("Middle:", dna[3:10])
print("Every second base:", dna[::2])

First base: A
Last base: C
First three: ATG
Middle: CGTACGT
Every second base: AGGAGTG


The returned sequence behavior is similar to Python strings, but the biological object is preserved for sequence slices.

This is especially useful when extracting:

- codons
- motifs
- exons
- domains
- subsequences

In [4]:
# Extract a subsequence using Python slicing.
region = dna[2:8]

# Display the extracted sequence and confirm its type.
print(region)
print(type(region))

GCGTAC
<class 'Bio.Seq.Seq'>


## 6. Basic Sequence Operations

`Seq` supports many familiar operations.

In [5]:
from Bio.Seq import Seq

# Create a sequence for basic string-like operations.
seq = Seq("ATGCGT")

# Search for a motif, count a nucleotide, and find a substring position.
print("Contains ATG:", "ATG" in seq)
print("Count of G:", seq.count("G"))
print("Position of CG:", seq.find("CG"))

# Concatenate two sequences and repeat a short sequence.
combined = Seq("ATG") + Seq("CCC")
repeated = Seq("AT") * 4

print("Combined:", combined)
print("Repeated:", repeated)

Contains ATG: True
Count of G: 2
Position of CG: 3
Combined: ATGCCC
Repeated: ATATATAT


These operations are useful, but remember:

- `count()` counts exact sequence occurrences.
- `find()` returns a position and does not perform biological alignment.
- concatenation does not validate biological compatibility.

# 7. DNA Complement

DNA base-pairing follows:

```text
A ↔ T
C ↔ G
```

Biopython provides:

```python
complement()
```

In [6]:
from Bio.Seq import Seq

# Create a DNA sequence.
dna = Seq("ATGCCGTA")

# Calculate the complement without reversing the sequence.
print("DNA:        ", dna)
print("Complement: ", dna.complement())

DNA:         ATGCCGTA
Complement:  TACGGCAT


The complement changes each base but keeps the original direction.

For many molecular-biology workflows, the **reverse complement** is more useful.

# 8. Reverse Complement

The reverse complement performs two operations:

1. Complement the bases.
2. Reverse the resulting sequence.

In [7]:
from Bio.Seq import Seq

# Create a DNA sequence.
dna = Seq("ATGCCGTA")

# Calculate the reverse complement of the DNA strand.
print("DNA:               ", dna)
print("Reverse complement:", dna.reverse_complement())

DNA:                ATGCCGTA
Reverse complement: TACGGCAT


### Biological interpretation

If a sequence is written in the 5' → 3' direction:

```text
5' - ATGCCGTA - 3'
```

its reverse complement is:

```text
5' - TACGGCAT - 3'
```

Reverse complements are fundamental when analyzing:

- double-stranded DNA
- primers
- restriction sites
- sequence motifs
- reads aligned to the opposite strand
- genomic regions on the reverse strand

# 9. DNA → RNA: Transcription

Biological transcription replaces thymine (`T`) with uracil (`U`).

Biopython provides:

```python
transcribe()
```

In [8]:
from Bio.Seq import Seq

# Create a DNA sequence.
dna = Seq("ATGCGTACG")

# Transcribe DNA into RNA by replacing T with U.
rna = dna.transcribe()

print("DNA:", dna)
print("RNA:", rna)

DNA: ATGCGTACG
RNA: AUGCGUACG


Conceptually:

```text
DNA:  ATGCGTACG
RNA:  AUGCGUACG
```

The sequence is still represented by a Biopython sequence object.

## RNA → DNA: Back Transcription

Biopython provides `back_transcribe()` for converting RNA back to a DNA-style sequence.

In [9]:
# Create an RNA sequence.
rna = Seq("AUGCGUACG")

# Convert the RNA representation back to a DNA-style sequence.
dna_again = rna.back_transcribe()

print("RNA:", rna)
print("DNA:", dna_again)

RNA: AUGCGUACG
DNA: ATGCGTACG


This operation is useful when a workflow switches between RNA and DNA representations.

It does **not** reconstruct biological history; it is a sequence representation conversion.

# 10. Translation

Translation converts a nucleotide sequence into an amino-acid sequence according to a genetic code.

In [10]:
from Bio.Seq import Seq

# Create a nucleotide sequence that will be translated.
coding_dna = Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG")

# Translate the nucleotide sequence using the default genetic code.
protein = coding_dna.translate()

print("DNA:    ", coding_dna)
print("Protein:", protein)

DNA:     ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG
Protein: MAIVMGR*KGAR*


The default translation uses the standard genetic code.

A nucleotide sequence is interpreted in groups of three:

```text
ATG GCC ATT GTA ...
 ↓   ↓   ↓   ↓
 M   A   I   V
```

Translation is frame-dependent. Starting from a different nucleotide changes the codon grouping.

## 10.1 Translation and Stop Codons

A stop codon is normally represented by `*`.

In [11]:
# Create a nucleotide sequence containing an in-frame stop codon.
seq = Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG")

# Translate the sequence and display stop codons as "*".
print(seq.translate())

MAIVMGR*KGAR*


You can stop translation at the first stop codon using `to_stop=True`.

In [12]:
# Translate only until the first in-frame stop codon.
protein_until_stop = seq.translate(to_stop=True)

print(protein_until_stop)

MAIVMGR


### Important distinction

`to_stop=True` does not find an open reading frame by itself.

It simply stops translation at the first in-frame stop codon.

ORF detection requires additional biological logic, such as identifying:

- a start codon
- a valid reading frame
- a downstream in-frame stop codon

## 10.2 Translation of a Complete CDS

If you know that a nucleotide sequence is a complete coding sequence, `cds=True` performs stricter validation.

Biopython checks biological constraints such as:

- valid start codon
- complete codons
- terminal stop codon

In [13]:
# Define a candidate complete coding sequence.
complete_cds = Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG")

try:
    # Translate while asking Biopython to validate the CDS structure.
    protein = complete_cds.translate(cds=True)
    print(protein)
except Exception as exc:
    # Report validation errors instead of stopping the workflow.
    print("CDS validation error:", exc)

CDS validation error: Extra in frame stop codon 'TGA' found.


The exact result depends on whether the sequence satisfies the rules of the selected genetic code.

This distinction is important in real pipelines:

```text
translate()
    ↓
general translation

translate(cds=True)
    ↓
translation with CDS validation
```

# 11. Alternative Genetic Codes

Not all organisms use exactly the same genetic code.

Biopython exposes codon tables through `Bio.Data.CodonTable`.

In [14]:
from Bio.Data import CodonTable

# Load the standard unambiguous DNA genetic code.
standard_table = CodonTable.unambiguous_dna_by_name["Standard"]

# Inspect start codons, stop codons, and mapped sense codons.
print("Start codons:", standard_table.start_codons)
print("Stop codons:", standard_table.stop_codons)
print("Number of codons:", len(standard_table.forward_table))

Start codons: ['TTG', 'CTG', 'ATG']
Stop codons: ['TAA', 'TAG', 'TGA']
Number of codons: 61


You can also inspect the table by numeric ID when working with known NCBI genetic-code identifiers.

Always verify the correct genetic code for the organism or biological system being analyzed.

In [15]:
from Bio.Seq import Seq

# Create the nucleotide sequence.
dna = Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG")

# Translate using NCBI genetic code table 1 (Standard).
protein_standard = dna.translate(table=1)

print("Standard code:", protein_standard)

Standard code: MAIVMGR*KGAR*


> **Practical rule:** never assume that the standard genetic code is appropriate for every biological dataset.

# 12. Reading Frames

A reading frame determines where codon grouping starts.

For a sequence:

```text
ATGCCGTA...
```

the three forward frames begin at positions:

```text
Frame +1 → ATG CGT ...
Frame +2 → TGC GTA ...
Frame +3 → GCC GTA ...
```

Biopython can translate a sequence after slicing it into a chosen frame.

In [16]:
# Define a DNA sequence for reading-frame analysis.
dna = Seq("ATGCCGTAACCGGATG")

# Translate the three possible forward reading frames.
for frame in range(3):
    # Shift the sequence start by the current frame offset.
    frame_seq = dna[frame:]
    protein = frame_seq.translate(to_stop=False)
    print(f"Frame +{frame + 1}: {protein}")

Frame +1: MP*PD
Frame +2: CRNRM
Frame +3: AVTG


E:\ANACONDA\Lib\site-packages\Bio\Seq.py:2877: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


This is a computational demonstration of frame dependence.

For real ORF discovery, also account for:

- start codons
- stop codons
- strand
- sequence completeness
- biological annotation
- organism-specific genetic codes

# 13. `MutableSeq`

A normal `Seq` object is designed for biological sequence representation and behaves like an immutable sequence.

When direct base editing is needed, Biopython provides `MutableSeq`.

In [17]:
from Bio.Seq import MutableSeq

# Create a mutable biological sequence.
mutable = MutableSeq("ATGCGT")

print("Original:", mutable)

# Replace the base at index 2 with A.
mutable[2] = "A"

print("Edited:", mutable)

Original: ATGCGT
Edited: ATACGT


This is useful for controlled computational edits.

For example:

```text
Original: ATGCGT
             ↓
Edit base
             ↓
Edited:   ATGAGT
```

You can convert between mutable and immutable representations.

In [18]:
from Bio.Seq import Seq, MutableSeq

# Create a mutable sequence that can be edited in place.
mutable = MutableSeq("ATGCGT")

# Convert MutableSeq into an immutable Seq object.
immutable = Seq(mutable)

print("Mutable:", mutable, type(mutable))
print("Seq:    ", immutable, type(immutable))

Mutable: ATGCGT <class 'Bio.Seq.MutableSeq'>
Seq:     ATGCGT <class 'Bio.Seq.Seq'>


### When should you use `MutableSeq`?

Use it when your algorithm genuinely needs repeated in-place edits.

For most analytical workflows, keeping sequences immutable is simpler and safer.

# 14. `SeqRecord`

A biological sequence often needs more than the sequence itself.

For example:

```text
Sequence
ID
Description
Source
Database references
Annotations
Features
Quality scores
```

`SeqRecord` packages these pieces together.

In [19]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

# Create a SeqRecord containing a biological sequence and metadata.
record = SeqRecord(
    Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG"),
    id="example_gene_001",
    name="example_gene",
    description="Example coding sequence"
)

# Inspect the main identifying fields.
print("ID:", record.id)
print("Name:", record.name)
print("Description:", record.description)
print("Sequence:", record.seq)

ID: example_gene_001
Name: example_gene
Description: Example coding sequence
Sequence: ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG


This is a major conceptual step:

```text
Seq
 ↓
sequence

SeqRecord
 ↓
sequence + biological identity + metadata
```

This becomes essential when working with FASTA, FASTQ, GenBank, and other biological file formats.

## 14.1 Adding Annotations

`SeqRecord` can store additional metadata.

In [20]:
# Add record-level metadata.
record.annotations["organism"] = "Example organism"
record.annotations["molecule_type"] = "DNA"
record.annotations["source"] = "Synthetic dataset"

# Display all stored annotations.
print(record.annotations)

{'organism': 'Example organism', 'molecule_type': 'DNA', 'source': 'Synthetic dataset'}


The exact annotation schema depends on the biological file format and workflow.

Do not invent biological metadata simply to fill fields. Metadata should be traceable to a source.

## 14.2 Database Cross-References

A sequence may correspond to identifiers in external databases.

`dbxrefs` can store those references.

In [21]:
# Add a cross-reference to an external database.
record.dbxrefs.append("ExampleDB:SEQ001")

# Display the stored database references.
print(record.dbxrefs)

['ExampleDB:SEQ001']


In real datasets, cross-references can point to resources such as:

- NCBI
- UniProt
- RefSeq
- Ensembl
- organism-specific databases

## 14.3 Letter Annotations

Per-position data can be stored in `letter_annotations`.

This is especially important for sequencing quality values.

In [22]:
# Store one Phred quality value for each sequence position.
record.letter_annotations["phred_quality"] = [40] * len(record)

# Inspect the first ten quality values.
print(record.letter_annotations["phred_quality"][:10])

[40, 40, 40, 40, 40, 40, 40, 40, 40, 40]


For FASTQ data, `phred_quality` is commonly used to represent per-base sequencing quality.

The length of a letter annotation must match the sequence length.

# 15. Slicing a `SeqRecord`

A `SeqRecord` can be sliced to extract a subsequence while preserving relevant record information.

In [23]:
# Create a sequence record with an identifier and description.
record = SeqRecord(
    Seq("ATGCGTACGTTAGC"),
    id="gene_001",
    description="Example gene"
)

# Extract a subsequence by slicing the record.
subrecord = record[3:10]

print("Original:", record.seq)
print("Subrecord:", subrecord.seq)
print("Subrecord ID:", subrecord.id)

Original: ATGCGTACGTTAGC
Subrecord: CGTACGT
Subrecord ID: gene_001


Slicing is useful for extracting regions such as:

- coding segments
- motifs
- domains
- genomic intervals
- experimental regions of interest

When metadata has positional meaning, always verify that the sliced record still represents the intended biological object.

# 16. Building a Small Biopython Sequence Toolkit

A good bioinformatics workflow should not repeat the same logic everywhere.

We can create reusable functions around Biopython objects.

In [24]:
from Bio.Seq import Seq

def sequence_summary(sequence):
    # Normalize the input and convert it to a Biopython Seq object.
    seq = Seq(str(sequence).upper())

    # Return common sequence statistics in a structured dictionary.
    return {
        "length": len(seq),
        "A": seq.count("A"),
        "T": seq.count("T"),
        "G": seq.count("G"),
        "C": seq.count("C"),
        "N": seq.count("N"),
        "gc_percent": (
            (seq.count("G") + seq.count("C")) / len(seq) * 100
            if len(seq) > 0 else 0.0
        ),
    }

# Run the reusable analyzer on an example sequence.
summary = sequence_summary("ATGCGTACGNN")

summary

{'length': 11,
 'A': 2,
 'T': 2,
 'G': 3,
 'C': 2,
 'N': 2,
 'gc_percent': 45.45454545454545}

The function accepts either a Python string or a sequence-like value and normalizes it into a `Seq`.

This pattern is useful when integrating Biopython with:

- pandas
- NumPy
- custom pipelines
- file parsers
- machine-learning preprocessing

## 16.1 A More Complete Analyzer

In [25]:
from Bio.Seq import Seq

def analyze_dna(sequence):
    # Normalize the input and create a Biopython Seq object.
    seq = Seq(str(sequence).upper())

    # Reject empty sequences because several statistics require a length > 0.
    if not seq:
        raise ValueError("Sequence cannot be empty.")

    # Define the accepted DNA alphabet and identify unexpected symbols.
    valid = set("ACGTN")
    invalid = sorted(set(seq) - valid)

    # Build a structured result containing measurements and transformations.
    result = {
        "sequence": seq,
        "length": len(seq),
        "valid": len(invalid) == 0,
        "invalid_symbols": invalid,
        "gc_percent": (
            (seq.count("G") + seq.count("C")) / len(seq) * 100
        ),
        "reverse_complement": seq.reverse_complement(),
        "rna": seq.transcribe(),
    }

    return result

# Analyze an example sequence containing an ambiguous base.
result = analyze_dna("ATGCGTACGNN")

# Print each result field.
for key, value in result.items():
    print(f"{key}: {value}")

sequence: ATGCGTACGNN
length: 11
valid: True
invalid_symbols: []
gc_percent: 45.45454545454545
reverse_complement: NNCGTACGCAT
rna: AUGCGUACGNN


Notice the separation between:

- representation (`Seq`)
- validation
- measurements
- transformations

This makes the function easier to test and reuse.

# 17. A Practical Translation Workflow

Let's build a small workflow that takes a coding sequence and produces useful biological outputs.

In [26]:
from Bio.Seq import Seq

# Define a coding DNA sequence for an end-to-end transformation workflow.
coding_sequence = Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG")

# Apply common biological sequence transformations.
print("DNA:", coding_sequence)
print("RNA:", coding_sequence.transcribe())
print("Protein:", coding_sequence.translate(to_stop=True))
print("Reverse complement:", coding_sequence.reverse_complement())
print("Length:", len(coding_sequence))

DNA: ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG
RNA: AUGGCCAUUGUAAUGGGCCGCUGAAAGGGUGCCCGAUAG
Protein: MAIVMGR
Reverse complement: CTATCGGGCACCCTTTCAGCGGCCCATTACAATGGCCAT
Length: 39


This is a small example of the typical transformation chain:

```text
DNA
 ↓ transcribe
RNA
 ↓ translate
Protein
```

and independently:

```text
DNA
 ↓ reverse complement
Opposite-strand representation
```

# 18. A Synthetic `SeqRecord` Dataset

To make the notebook reproducible, we can create several sequence records in memory.

In [27]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

# Build a small in-memory dataset of biological sequence records.
records = [
    SeqRecord(
        Seq("ATGCGTACGTTAGC"),
        id="gene_001",
        description="Synthetic gene 1"
    ),
    SeqRecord(
        Seq("ATGGCCATTGTAATG"),
        id="gene_002",
        description="Synthetic gene 2"
    ),
    SeqRecord(
        Seq("GGGCGCGCGCGTAA"),
        id="gene_003",
        description="GC-rich synthetic gene"
    ),
]

# Inspect the ID, length, and sequence for every record.
for record in records:
    print(record.id, len(record), record.seq)

gene_001 14 ATGCGTACGTTAGC
gene_002 15 ATGGCCATTGTAATG
gene_003 14 GGGCGCGCGCGTAA


## 18.1 Batch Analysis

In [28]:
def gc_percent(seq):
    # Normalize the input and convert it to a Biopython Seq object.
    seq = Seq(str(seq).upper())

    # Return GC percentage while handling an empty sequence safely.
    return (
        (seq.count("G") + seq.count("C")) / len(seq) * 100
        if len(seq) else 0.0
    )

batch_results = []

# Analyze every sequence record and collect structured results.
for record in records:
    batch_results.append({
        "id": record.id,
        "length": len(record.seq),
        "gc_percent": gc_percent(record.seq),
        "reverse_complement": str(record.seq.reverse_complement()),
    })

batch_results

[{'id': 'gene_001',
  'length': 14,
  'gc_percent': 50.0,
  'reverse_complement': 'GCTAACGTACGCAT'},
 {'id': 'gene_002',
  'length': 15,
  'gc_percent': 40.0,
  'reverse_complement': 'CATTACAATGGCCAT'},
 {'id': 'gene_003',
  'length': 14,
  'gc_percent': 78.57142857142857,
  'reverse_complement': 'TTACGCGCGCGCCC'}]

This is the beginning of a real batch-processing workflow.

Later, `SeqIO.parse()` will allow the same logic to operate directly on FASTA and FASTQ files.

# 19. Biological Interpretation vs Computational Measurement

Biopython can calculate a value, but the value still needs biological interpretation.

For example:

```python
gc_percent(sequence)
```

computes GC percentage.

It does **not** automatically tell us:

- whether the sequence is biologically functional
- whether it belongs to a gene
- whether it is pathogenic
- whether a high GC value is biologically important
- whether the sequence is correctly assembled

A strong bioinformatics workflow separates:

```text
Computational measurement
        ↓
Quality control
        ↓
Biological context
        ↓
Interpretation
```

This distinction is essential when building research-grade pipelines.

# 20. Common Biopython Mistakes

### Mistake 1 — Treating translation as ORF prediction

```python
seq.translate()
```

does not automatically identify biologically meaningful ORFs.

---

### Mistake 2 — Ignoring reading frames

Translation depends on the starting position.

---

### Mistake 3 — Assuming the standard genetic code

Some organisms and organelles use alternative genetic codes.

---

### Mistake 4 — Confusing complement with reverse complement

They are different operations.

---

### Mistake 5 — Treating `find()` as alignment

`find()` performs exact substring searching.

It does not account for:

- substitutions
- insertions
- deletions
- gaps

---

### Mistake 6 — Inventing metadata

Annotations should come from a trustworthy source.

---

### Mistake 7 — Using `MutableSeq` unnecessarily

If no in-place editing is required, `Seq` is often the simpler representation.

# 21. Mini-Project — Biopython Sequence Analyzer

## Objective

Build a reusable analyzer that accepts DNA sequences and reports:

- sequence length
- nucleotide counts
- GC percentage
- reverse complement
- RNA sequence
- translated protein
- number of stop symbols in the translated sequence

In [29]:
from Bio.Seq import Seq

def biopython_sequence_analyzer(sequence, to_stop=False):
    # Normalize the input and create a Biopython Seq object.
    seq = Seq(str(sequence).upper())

    # Reject empty input before calculating sequence statistics.
    if len(seq) == 0:
        raise ValueError("Sequence cannot be empty.")

    # Check whether all symbols belong to the accepted DNA alphabet.
    allowed = set("ACGTN")
    invalid = sorted(set(seq) - allowed)

    # Generate RNA and protein representations.
    rna = seq.transcribe()
    protein = seq.translate(to_stop=to_stop)

    # Return a structured dictionary that can be reused downstream.
    return {
        "sequence": str(seq),
        "length": len(seq),
        "valid_symbols": len(invalid) == 0,
        "invalid_symbols": invalid,
        "A": seq.count("A"),
        "C": seq.count("C"),
        "G": seq.count("G"),
        "T": seq.count("T"),
        "N": seq.count("N"),
        "gc_percent": (
            (seq.count("G") + seq.count("C")) / len(seq) * 100
        ),
        "rna": str(rna),
        "reverse_complement": str(seq.reverse_complement()),
        "protein": str(protein),
        "stop_count": str(protein).count("*"),
    }

# Run the analyzer on an example coding sequence.
analysis = biopython_sequence_analyzer(
    "ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG"
)

# Print each field of the resulting report.
for key, value in analysis.items():
    print(f"{key}: {value}")

sequence: ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG
length: 39
valid_symbols: True
invalid_symbols: []
A: 9
C: 8
G: 14
T: 8
N: 0
gc_percent: 56.41025641025641
rna: AUGGCCAUUGUAAUGGGCCGCUGAAAGGGUGCCCGAUAG
reverse_complement: CTATCGGGCACCCTTTCAGCGGCCCATTACAATGGCCAT
protein: MAIVMGR*KGAR*
stop_count: 2


### Mini-Project Extension

Extend the analyzer so that it can:

1. accept a list of sequences;
2. assign each sequence an ID;
3. calculate summary statistics;
4. filter sequences by minimum length;
5. filter sequences by GC range;
6. translate all retained sequences;
7. return a list of dictionaries suitable for a pandas DataFrame;
8. write the final sequences to FASTA in the next notebook using `SeqIO`.

The final point intentionally introduces the next stage of the Biopython workflow.

# 22. Exercises

## Exercise 1 — Sequence Transformation

Create a `Seq` object and calculate:

- complement
- reverse complement
- RNA
- protein

---

## Exercise 2 — Reading Frames

Given a DNA sequence, translate all three forward reading frames.

---

## Exercise 3 — Quality Control

Write a function that:

- accepts a DNA sequence
- checks whether all symbols belong to `ACGTN`
- reports invalid symbols
- calculates GC percentage

---

## Exercise 4 — Mutable Sequence

Create a `MutableSeq`, introduce a controlled point mutation, and convert it back to `Seq`.

---

## Exercise 5 — SeqRecord

Create a `SeqRecord` with:

- ID
- name
- description
- organism annotation
- molecule type
- a database cross-reference

---

## Exercise 6 — Batch Analysis

Analyze at least five sequences and produce a summary containing:

- ID
- length
- GC percentage
- reverse complement
- translated sequence

---

## Exercise 7 — CDS Validation

Create several nucleotide sequences and test which ones satisfy `translate(cds=True)` under the standard genetic code.

Explain why invalid sequences fail.

# 23. Capstone — Biopython Sequence Processing Toolkit

Build a small reusable toolkit with the following components:

```text
Input sequence
      ↓
Normalization
      ↓
Validation
      ↓
Seq object
      ↓
Basic statistics
      ↓
GC analysis
      ↓
Reverse complement
      ↓
Transcription
      ↓
Translation
      ↓
SeqRecord
      ↓
Structured report
```

Your toolkit should expose functions such as:

```python
validate_dna()
calculate_gc()
reverse_complement()
transcribe_dna()
translate_dna()
create_seqrecord()
analyze_sequence()
```

### Portfolio goal

The project should be written as reusable Python code rather than one long notebook cell.

A strong implementation should include:

- docstrings
- input validation
- clear function names
- predictable return values
- informative errors
- small test cases
- biological comments where appropriate

# 24. Biopython Cheat Sheet

This section is designed to remain useful after finishing the notebook.

## Core Imports

```python
from Bio.Seq import Seq, MutableSeq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
from Bio.SeqFeature import SeqFeature, FeatureLocation
from Bio.Data import CodonTable
```

## `Seq` Cheat Sheet

| Method / Operation | Purpose | Example |
|---|---|---|
| `Seq()` | Create a sequence | `Seq("ATGC")` |
| `len()` | Sequence length | `len(seq)` |
| `seq[i]` | Indexing | `seq[0]` |
| `seq[a:b]` | Slicing | `seq[2:8]` |
| `seq.count()` | Count exact pattern/base | `seq.count("GC")` |
| `seq.find()` | Find exact substring | `seq.find("ATG")` |
| `seq.complement()` | DNA complement | `seq.complement()` |
| `seq.reverse_complement()` | Reverse complement | `seq.reverse_complement()` |
| `seq.transcribe()` | DNA → RNA | `seq.transcribe()` |
| `seq.back_transcribe()` | RNA → DNA | `rna.back_transcribe()` |
| `seq.translate()` | DNA/RNA → protein | `seq.translate()` |
| `seq.translate(to_stop=True)` | Translate until first stop | `seq.translate(to_stop=True)` |
| `seq.translate(cds=True)` | Validate and translate CDS | `seq.translate(cds=True)` |
| `seq.upper()` | Uppercase | `seq.upper()` |
| `seq.lower()` | Lowercase | `seq.lower()` |
| `"ATG" in seq` | Membership test | `"ATG" in seq` |
| `seq1 + seq2` | Concatenate sequences | `seq1 + seq2` |
| `seq * n` | Repeat sequence | `seq * 3` |

## `MutableSeq` Cheat Sheet

| Operation | Purpose | Example |
|---|---|---|
| `MutableSeq()` | Create mutable sequence | `MutableSeq("ATGC")` |
| `mutable[i] = "A"` | Edit one position | `seq[2] = "A"` |
| `mutable.append()` | Add a base | `seq.append("G")` |
| `mutable.extend()` | Add multiple bases | `seq.extend("AT")` |
| `mutable.pop()` | Remove and return item | `seq.pop()` |
| `mutable.remove()` | Remove a value | `seq.remove("A")` |
| `mutable.reverse()` | Reverse in place | `seq.reverse()` |
| `Seq(mutable)` | Convert to `Seq` | `seq.toseq()` |

## `SeqRecord` Cheat Sheet

```python
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq
```

| Attribute | Purpose |
|---|---|
| `record.seq` | Biological sequence |
| `record.id` | Primary identifier |
| `record.name` | Short name |
| `record.description` | Human-readable description |
| `record.annotations` | Record-level metadata |
| `record.dbxrefs` | Database cross-references |
| `record.features` | Biological features |
| `record.letter_annotations` | Per-position data |
| `record.seq` | Access the underlying `Seq` |

Example:

```python
record = SeqRecord(
    Seq("ATGC"),
    id="gene_001",
    name="gene",
    description="Example sequence"
)

record.annotations["organism"] = "Example organism"
record.dbxrefs.append("DB:001")
```

## Translation Cheat Sheet

```python
seq.translate()
```

General translation.

```python
seq.translate(to_stop=True)
```

Stop at the first in-frame stop codon.

```python
seq.translate(cds=True)
```

Validate that the sequence behaves like a complete CDS under the selected genetic code.

```python
seq.translate(table=1)
```

Use a specific genetic-code table.

```python
seq.translate(stop_symbol="*")
```

Control the symbol used for stop codons.

### Important

Translation depends on:

- reading frame
- genetic code
- sequence completeness
- strand orientation
- biological context

## Codon Tables

```python
from Bio.Data import CodonTable
```

Common access patterns:

```python
CodonTable.unambiguous_dna_by_name["Standard"]
CodonTable.unambiguous_dna_by_id[1]
```

Useful attributes include:

```python
table.start_codons
table.stop_codons
table.forward_table
table.back_table
```

## `SeqIO` Preview

Full `SeqIO` workflows are covered in the next notebook.

```python
from Bio import SeqIO
```

Core functions:

| Function | Purpose |
|---|---|
| `SeqIO.parse()` | Read multiple records |
| `SeqIO.read()` | Read exactly one record |
| `SeqIO.write()` | Write records |
| `SeqIO.convert()` | Convert between supported formats |

Typical pattern:

```python
records = SeqIO.parse("input.fasta", "fasta")

for record in records:
    print(record.id)
    print(record.seq)
```

Write records:

```python
SeqIO.write(records, "output.fasta", "fasta")
```

## `SeqFeature` Preview

```python
from Bio.SeqFeature import SeqFeature, FeatureLocation
```

Typical concepts:

```python
location = FeatureLocation(10, 50, strand=1)

feature = SeqFeature(
    location=location,
    type="CDS"
)
```

Important concepts:

- `FeatureLocation`
- `SeqFeature`
- `feature.type`
- `feature.location`
- `feature.qualifiers`
- `feature.extract(record.seq)`

These become especially important when working with GenBank files.

## Frequently Used Biopython Modules

| Module | Main use |
|---|---|
| `Bio.Seq` | Sequence manipulation |
| `Bio.SeqRecord` | Sequence + metadata |
| `Bio.SeqIO` | Biological file I/O |
| `Bio.SeqFeature` | Features and annotations |
| `Bio.SeqUtils` | Sequence/protein utilities |
| `Bio.Data.CodonTable` | Genetic codes |
| `Bio.Align` | Alignment |
| `Bio.Blast` | BLAST result handling |
| `Bio.Entrez` | NCBI services |
| `Bio.Phylo` | Phylogenetics |
| `Bio.Restriction` | Restriction enzymes |

# 25. Most-Used Biopython Commands — Quick Reference

```python
# Sequence
from Bio.Seq import Seq

seq = Seq("ATGGCC")

len(seq)
seq[0]
seq[:3]
seq.count("G")
seq.find("ATG")

seq.complement()
seq.reverse_complement()

seq.transcribe()
seq.back_transcribe()

seq.translate()
seq.translate(to_stop=True)
seq.translate(cds=True)

# Mutable sequence
from Bio.Seq import MutableSeq

mutable = MutableSeq("ATGC")
mutable[0] = "G"
seq = Seq(mutable)

# Sequence record
from Bio.SeqRecord import SeqRecord

record = SeqRecord(
    seq,
    id="sequence_001",
    description="Example sequence"
)

record.annotations
record.dbxrefs
record.features
record.letter_annotations

# File I/O — introduced next
from Bio import SeqIO

SeqIO.parse(...)
SeqIO.read(...)
SeqIO.write(...)
SeqIO.convert(...)

# Features
from Bio.SeqFeature import SeqFeature, FeatureLocation

location = FeatureLocation(10, 50, strand=1)
feature = SeqFeature(location=location, type="CDS")

# Codon tables
from Bio.Data import CodonTable

table = CodonTable.unambiguous_dna_by_name["Standard"]
```

# 26. What We Can Do Now

After this notebook, the basic conceptual chain is:

```text
Python string
     ↓
Seq
     ↓
SeqRecord
     ↓
SeqIO
     ↓
FASTA / FASTQ / GenBank
     ↓
Features / annotations
     ↓
Alignment / BLAST / NCBI / phylogenetics
```

The next step is to move from in-memory objects to **real biological files**.

### Next Notebook

**04 — FASTA & FASTQ Processing with Biopython**

We will work with:

- `SeqIO.parse()`
- `SeqIO.read()`
- `SeqIO.write()`
- FASTA
- FASTQ
- quality scores
- filtering
- batch processing
- format conversion
- practical sequence-processing pipelines

The important transition is:

> **From manipulating sequences → to processing real biological datasets.**

# Key Takeaways

- `Seq` is the core Biopython sequence object.
- `MutableSeq` is useful for controlled in-place sequence editing.
- `SeqRecord` combines a sequence with biological metadata.
- `complement()` and `reverse_complement()` are different operations.
- `transcribe()` models DNA → RNA sequence conversion.
- `back_transcribe()` models RNA → DNA representation.
- `translate()` converts nucleotide sequences into amino-acid sequences according to a genetic code.
- Translation is frame-dependent and is not the same thing as ORF prediction.
- `cds=True` provides stricter CDS validation.
- Alternative genetic codes must be considered when biologically appropriate.
- Biopython objects are most powerful when combined with reproducible workflows and biological context.
- The next major step is `SeqIO` and real FASTA/FASTQ/GenBank data.